In [1]:
from google.cloud import bigquery

# The client will now automatically find your credentials via ADC
client = bigquery.Client(project="methodical-mark-493108-d4")
print("Authenticated via ADC")

Authenticated via ADC


In [2]:
import pandas as pd
def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query on BigQuery and return a pandas DataFrame."""
    job = client.query(sql)
    return job.to_dataframe()

In [3]:
df_sepsis = run_query("""
SELECT * FROM `physionet-data.mimiciv_3_1_derived.sepsis3` LIMIT 1000;
""")
df_sepsis.head()

c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_id,stay_id,antibiotic_time,culture_time,suspected_infection_time,sofa_time,sofa_score,respiration,coagulation,liver,cardiovascular,cns,renal,sepsis3
0,18212223,30752269,2136-08-15 08:00:00,2136-08-14 23:00:00,2136-08-14 23:00:00,2136-08-15 04:00:00,2,0,0,0,0,0,2,True
1,13736311,39454408,2178-04-29 08:00:00,2178-04-28 15:45:00,2178-04-28 15:45:00,2178-04-29 09:00:00,2,0,0,0,0,1,1,True
2,19085966,34095671,2156-02-03 04:00:00,2156-02-03 03:52:00,2156-02-03 03:52:00,2156-02-03 17:00:00,2,1,0,0,0,1,0,True
3,10147182,33175266,2179-11-23 00:00:00,2179-11-22 21:08:00,2179-11-22 21:08:00,2179-11-22 21:00:00,2,2,0,0,0,0,0,True
4,19669410,38510130,2143-10-02 20:00:00,2143-10-03 00:19:00,2143-10-02 20:00:00,2143-10-02 19:00:00,2,0,0,0,0,2,0,True


In [ ]:
import torch

In [3]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA Capability: (8, 9)


In [16]:
import json
import pandas as pd
import numpy as np

df = pd.read_csv('full_data.csv') 

mira_data = []
# GROUP BY BOTH STAY AND ITEM
for (stay_id, itemid), group in df.groupby(['stay_id', 'itemid']):
    group = group.sort_values('hrs_since_anchor')
    
    # MIRA needs to know WHAT this variable is. 
    # We use 'feat_id' so the Graph-RAG can identify it later.
    vals = group['valuenum'].fillna(0.0).tolist()
    times = group['hrs_since_anchor'].tolist()
    mask = [1] * len(vals)

    patient_var_entry = {
        "stay_id": str(stay_id),
        "itemid": str(itemid), # CRITICAL for Graph-RAG lookup
        "label": 1 if str(group['label'].iloc[0]).lower() == 'true' else 0,
        "sequence": vals,
        "time": times,
        "mask": mask
    }
    mira_data.append(patient_var_entry)

with open('train_mira_kaggle.jsonl', 'w') as f:
    for entry in mira_data:
        f.write(json.dumps(entry) + '\n')

In [2]:
import torch
import json
from mira.mira.models.modeling_mira import MIRAForPrediction

# 1. Load your local model
model_path = "mira/checkpoints"
model = MIRAForPrediction.from_pretrained(model_path).cuda()
model.eval()

c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0424 10:30:20.537000 27872 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


MIRAForPrediction(
  (model): MIRAModel(
    (embed_layer): MIRAInputEmbedding(
      (emb_layer): Linear(in_features=1, out_features=384, bias=False)
      (gate_layer): Linear(in_features=1, out_features=384, bias=False)
      (act_fn): SiLU()
    )
    (layers): ModuleList(
      (0-11): 12 x MIRADecoderLayer(
        (self_attn): MIRAAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=False)
          (rotary_emb): ContinuousTimeRotaryEmbedding()
        )
        (ffn_layer): MIRASparseExpertsLayer(
          (gate): Linear(in_features=384, out_features=8, bias=False)
          (experts): ModuleList(
            (0-7): 8 x MIRATemporalBlock(
              (gate_proj): Linear(in_features=384, out_features=768, bias=False)
              (up_p

In [3]:
import torch
import json
import numpy as np
from mira.mira.models.modeling_mira import MIRAForPrediction
from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP & MAPPINGS
# Using the itemid dictionary you provided earlier
ITEM_MAP = {
    "221289": "Epinephrine", "221662": "Dopamine", "221749": "Phenylephrine",
    "221906": "Norepinephrine", "222315": "Vasopressin", "220210": "Resp Rate",
    "220277": "SpO2", "220045": "Heart Rate", "220052": "MAP", "223762": "Temp (C)"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "mira/checkpoints" # Point this to your saved folder
model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

def run_forecast(itemid, history_vals, history_times, steps_to_forecast=3):
    """
    Takes a history of one variable and predicts the future trajectory.
    """
    label = ITEM_MAP.get(itemid, f"Unknown ({itemid})")
    
    # Convert to Tensors [Batch=1, Length]
    seq = torch.tensor([history_vals], dtype=torch.float32).to(device)
    time = torch.tensor([history_times], dtype=torch.float32).to(device)
    attn_mask = torch.ones_like(seq).to(device)

    # 2. TIME NORMALIZATION (Required for MIRA's CT-RoPE)
    # MIRA needs relative time geometry to understand the gaps
    full_scaled_times, _, _ = normalize_time_for_ctrope(
        time_values=time,
        attention_mask=attn_mask,
        seq_length=seq.shape[1],
        alpha=1.0
    )

    # 3. AUTOREGRESSIVE FORECASTING
    current_vals = seq.clone()
    current_times = full_scaled_times.clone()
    predictions = []

    print(f"\n--- Forecaster Input: {label} ---")
    print(f"Recent History: {history_vals[-3:]} at times {history_times[-3:]}")

    with torch.no_grad():
        for i in range(steps_to_forecast):
            # MIRA expects [Batch, Length, 1] for the values
            inp_vals = current_vals.unsqueeze(-1)
            
            output = model(
                input_ids=inp_vals,
                time_values=current_times,
                return_dict=True
            )
            
            # Get the very last logit (the future prediction)
            next_val = output.logits[:, -1, :]
            predictions.append(next_val.item())

            # Update tensors for the next step in the loop
            current_vals = torch.cat([current_vals, next_val], dim=1)
            
            # Simple time-step increment for the forecast (e.g., +1 hour)
            next_time = current_times[:, -1:] + 1.0 
            current_times = torch.cat([current_times, next_time], dim=1)

    print(f"Predicted Trend: {predictions}")
    return predictions



In [14]:
l = {}
with open('train_mira.jsonl', 'r') as f:
    for line in f:
        entry = json.loads(line)
        itemid = entry['itemid']
        history_vals = entry['sequence']
        history_times = entry['time']
        if itemid not in l:
            l[itemid] = (history_vals, history_times)


for itemid, (history_vals, history_times) in l.items():
    run_forecast(itemid, history_vals, history_times, steps_to_forecast=3)


--- Forecaster Input: Unknown (50885) ---
Recent History: [0.6] at times [0.3333333333333333]
Predicted Trend: [0.5363202691078186, 0.4284543991088867, 0.3084966540336609]

--- Forecaster Input: Unknown (50912) ---
Recent History: [1.0, 0.9] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [0.761544942855835, 0.613304078578949, 0.4709845185279846]

--- Forecaster Input: Unknown (50931) ---
Recent History: [132.0, 119.0] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [14.087804794311523, 5.6179704666137695, 1.2159326076507568]

--- Forecaster Input: Unknown (51265) ---
Recent History: [260.0, 251.0] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [1.7128721475601196, 1.8138915300369263, 2.122591018676758]

--- Forecaster Input: Unknown (51301) ---
Recent History: [21.9, 22.5] at times [-2.966666666666667, 0.3333333333333333]
Predicted Trend: [15.647808074951172, 5.811712741851807, 2.297614574432373]

--- Forecaster Input: Heart R

In [11]:
run_forecast("220045", [72], [0], steps_to_forecast=3)


--- Forecaster Input: Heart Rate ---
Recent History: [72] at times [0]
Predicted Trend: [15.300379753112793, 3.735386848449707, 1.365187644958496]


[15.300379753112793, 3.735386848449707, 1.365187644958496]

In [12]:
# import torch
# import json
# import numpy as np
# import pandas as pd
# from mira.mira.models.modeling_mira import MIRAForPrediction
# from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "mira/checkpoints" 
model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

# Mapping for the output
ITEM_MAP = {
    "221289": "Epinephrine", "221662": "Dopamine", "221749": "Phenylephrine",
    "221906": "Norepinephrine", "222315": "Vasopressin", "220210": "Resp Rate",
    "220277": "SpO2", "220045": "Heart Rate", "220052": "MAP", "223762": "Temp (C)"
}

def run_distinct_forecasts(data_path, steps=3):
    seen_items = set()
    
    with open(data_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            itemid = str(data.get('itemid'))
            
            # Only process each itemid once for the demo
            if itemid in seen_items or itemid not in ITEM_MAP:
                continue
            seen_items.add(itemid)
            
            # Prepare Data
            seq = torch.tensor([data['sequence']], dtype=torch.float32).to(device)
            times = torch.tensor([data['time']], dtype=torch.float32).to(device)
            
            # Stats for Normalization (Crucial for MIRA)
            mean = seq.mean()
            std = seq.std() + 1e-6
            seq_norm = (seq - mean) / std

            # Time Normalization
            full_scaled_times, _, _ = normalize_time_for_ctrope(
                time_values=times,
                attention_mask=torch.ones_like(times),
                seq_length=seq.shape[1],
                alpha=1.0
            )

            # Autoregressive Forecast
            cur_vals = seq_norm.clone()
            cur_times = full_scaled_times.clone()
            preds_norm = []

            with torch.no_grad():
                for _ in range(steps):
                    out = model(input_ids=cur_vals.unsqueeze(-1), time_values=cur_times)
                    next_val_norm = out.logits[:, -1, :]
                    preds_norm.append(next_val_norm.item())
                    
                    # Update for next step
                    cur_vals = torch.cat([cur_vals, next_val_norm], dim=1)
                    next_t = cur_times[:, -1:] + 1.0 # Predict 1 hour ahead
                    cur_times = torch.cat([cur_times, next_t], dim=1)

            # De-normalize
            preds_real = [round((p * std.item()) + mean.item(), 2) for p in preds_norm]
            history = [round(x, 2) for x in data['sequence'][-3:]]

            print(f"[{ITEM_MAP[itemid]}] ID: {itemid}")
            print(f"  > History (last 3): {history}")
            print(f"  > MIRA Forecast:    {preds_real}")
            print("-" * 40)

# 2. EXECUTE
print(f"Running forecasts on unique items from training data...\n")
run_distinct_forecasts("./train_mira.jsonl")

Running forecasts on unique items from training data...

[Heart Rate] ID: 220045
  > History (last 3): [76.0, 77.0, 83.0]
  > MIRA Forecast:    [81.27, 82.33, 84.72]
----------------------------------------
[Resp Rate] ID: 220210
  > History (last 3): [20.0, 17.0, 31.0]
  > MIRA Forecast:    [21.35, 19.03, 18.52]
----------------------------------------
[SpO2] ID: 220277
  > History (last 3): [97.0, 99.0, 97.0]
  > MIRA Forecast:    [98.22, 98.39, 98.44]
----------------------------------------
[MAP] ID: 220052
  > History (last 3): [85.0, 79.0, 81.0]
  > MIRA Forecast:    [82.94, 85.62, 87.67]
----------------------------------------
[Phenylephrine] ID: 221749
  > History (last 3): [2.0, 1.0, 0.5]
  > MIRA Forecast:    [1.82, 1.65, 1.46]
----------------------------------------
[Norepinephrine] ID: 221906
  > History (last 3): [0.03, 0.05, 0.07]
  > MIRA Forecast:    [0.07, 0.06, 0.05]
----------------------------------------
[Temp (C)] ID: 223762
  > History (last 3): [37.3, 37.2, 37

C:\Users\nisha\AppData\Local\Temp\ipykernel_27872\253878398.py:40: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = seq.std() + 1e-6


[Dopamine] ID: 221662
  > History (last 3): [5.0, 6.01, 5.0]
  > MIRA Forecast:    [5.24, 5.32, 5.36]
----------------------------------------


In [13]:
# import numpy as np

# class ClinicalSymbolicInterpreter:
#     """
#     Implements symbolic clinical logic to interpret neural forecasts 
#     from the MIRA encoder[cite: 19, 81].
#     """
#     def __init__(self):
#         # Clinical thresholds based on Sepsis-3 / qSOFA standards [cite: 28]
#         self.thresholds = {
#             "220052": {"label": "MAP", "min": 65, "unit": "mmHg"},
#             "220045": {"label": "Heart Rate", "max": 100, "unit": "bpm"},
#             "220210": {"label": "Resp Rate", "max": 22, "unit": "bpm"},
#             "220277": {"label": "SpO2", "min": 90, "unit": "%"},
#             "223762": {"label": "Temperature", "range": (36, 38), "unit": "C"}
#         }
        
#         # Mapping pressors for shock logic [cite: 106]
#         self.vasopressors = ["221289", "221662", "221749", "221906", "222315"]

#     def evaluate_forecast(self, itemid, predicted_values, current_meds=None):
#         """
#         Maps raw signals with structured medical logic[cite: 19].
#         """
#         alerts = []
#         item_meta = self.thresholds.get(itemid)
        
#         if not item_meta:
#             return None

#         last_forecast = predicted_values[-1]
#         label = item_meta["label"]

#         # 1. Check for threshold violations (Symbolic Logic) [cite: 28]
#         if "min" in item_meta and last_forecast < item_meta["min"]:
#             alerts.append(f"CRITICAL: Low {label} ({last_forecast:.1f} {item_meta['unit']})")
        
#         if "max" in item_meta and last_forecast > item_meta["max"]:
#             alerts.append(f"WARNING: Elevated {label} ({last_forecast:.1f} {item_meta['unit']})")

#         # 2. Complex Sepsis-3 Reasoning: Refractory Shock [cite: 44, 106]
#         if itemid == "220052" and last_forecast < 65:
#             is_on_pressors = any(p in (current_meds or []) for p in self.vasopressors)
#             if is_on_pressors:
#                 alerts.append("ALERT: Predicted MAP < 65 while on vasopressors. Indicates Refractory Septic Shock.")

#         return alerts

# # --- INTEGRATION EXAMPLE ---
# # Assume MIRA predicts these MAP values for the next 3 hours
# mira_forecast = [68.2, 64.5, 61.3] 
# patient_meds = ["221906"] # Patient is on Norepinephrine

# interpreter = ClinicalSymbolicInterpreter()
# symbolic_alerts = interpreter.evaluate_forecast("220052", mira_forecast, patient_meds)

# print(f"--- Neurosymbolic Clinical Output ---")
# for alert in symbolic_alerts:
#     print(f"[*] {alert}")

--- Neurosymbolic Clinical Output ---
[*] CRITICAL: Low MAP (61.3 mmHg)
[*] ALERT: Predicted MAP < 65 while on vasopressors. Indicates Refractory Septic Shock.


In [19]:
# import torch
# import json
# import numpy as np
# import pandas as pd
# from mira.mira.models.modeling_mira import MIRAForPrediction
# from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "mira/checkpoints" 
model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

# Mapping for the output
# ITEM_MAP = {
#     "221289": "Epinephrine", "221662": "Dopamine", "221749": "Phenylephrine",
#     "221906": "Norepinephrine", "222315": "Vasopressin", "220210": "Resp Rate",
#     "220277": "SpO2", "220045": "Heart Rate", "220052": "MAP", "223762": "Temp (C)"
# }

ITEM_MAP = {
    "220052": "MAP",
    "220045": "Heart Rate",
    "220210": "Resp Rate",
    "220277": "SpO2",
    "223762": "Temp (C)",
    "50813": "Lactate",
    "50820": "pH",
    "50912": "Creatinine",
    "50885": "Bilirubin",
    "51265": "Platelets",
    "51301": "WBC",
    "50931": "Glucose"
}

def run_distinct_forecasts(data_path, steps=3):
    seen_items = set()
    
    with open(data_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            itemid = str(data.get('itemid'))
            
            # Only process each itemid once for the demo
            if itemid in seen_items or itemid not in ITEM_MAP:
                continue
            seen_items.add(itemid)
            
            # Prepare Data
            seq = torch.tensor([data['sequence']], dtype=torch.float32).to(device)
            times = torch.tensor([data['time']], dtype=torch.float32).to(device)
            
            # Stats for Normalization (Crucial for MIRA)
            mean = seq.mean()
            std = seq.std() + 1e-6
            seq_norm = (seq - mean) / std

            # Time Normalization
            full_scaled_times, _, _ = normalize_time_for_ctrope(
                time_values=times,
                attention_mask=torch.ones_like(times),
                seq_length=seq.shape[1],
                alpha=1.0
            )

            # Autoregressive Forecast
            cur_vals = seq_norm.clone()
            cur_times = full_scaled_times.clone()
            preds_norm = []

            with torch.no_grad():
                for _ in range(steps):
                    out = model(input_ids=cur_vals.unsqueeze(-1), time_values=cur_times)
                    next_val_norm = out.logits[:, -1, :]
                    preds_norm.append(next_val_norm.item())
                    
                    # Update for next step
                    cur_vals = torch.cat([cur_vals, next_val_norm], dim=1)
                    next_t = cur_times[:, -1:] + 1.0 # Predict 1 hour ahead
                    cur_times = torch.cat([cur_times, next_t], dim=1)

            # De-normalize
            preds_real = [round((p * std.item()) + mean.item(), 2) for p in preds_norm]
            history = [round(x, 2) for x in data['sequence'][-3:]]

            print(f"[{ITEM_MAP[itemid]}] ID: {itemid}")
            print(f"  > History (last 3): {history}")
            print(f"  > MIRA Forecast:    {preds_real}")
            print("-" * 40)

# 2. EXECUTE
print(f"Running forecasts on unique items from training data...\n")
run_distinct_forecasts("./train_mira.jsonl")

Running forecasts on unique items from training data...

[Bilirubin] ID: 50885
  > History (last 3): [0.6]
  > MIRA Forecast:    [nan, nan, nan]
----------------------------------------


C:\Users\nisha\AppData\Local\Temp\ipykernel_27872\3199800900.py:55: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = seq.std() + 1e-6


[Creatinine] ID: 50912
  > History (last 3): [1.0, 0.9]
  > MIRA Forecast:    [0.92, 0.94, 0.94]
----------------------------------------
[Glucose] ID: 50931
  > History (last 3): [132.0, 119.0]
  > MIRA Forecast:    [121.68, 123.63, 124.42]
----------------------------------------
[Platelets] ID: 51265
  > History (last 3): [260.0, 251.0]
  > MIRA Forecast:    [252.85, 254.2, 254.75]
----------------------------------------
[WBC] ID: 51301
  > History (last 3): [21.9, 22.5]
  > MIRA Forecast:    [22.38, 22.25, 22.24]
----------------------------------------
[Heart Rate] ID: 220045
  > History (last 3): [76.0, 77.0, 83.0]
  > MIRA Forecast:    [81.27, 82.33, 84.72]
----------------------------------------
[Resp Rate] ID: 220210
  > History (last 3): [20.0, 17.0, 31.0]
  > MIRA Forecast:    [21.35, 19.03, 18.52]
----------------------------------------
[SpO2] ID: 220277
  > History (last 3): [97.0, 99.0, 97.0]
  > MIRA Forecast:    [98.22, 98.39, 98.44]
---------------------------------

In [23]:
import torch
import json
import os
from mira.mira.models.modeling_mira import MIRAForPrediction
from mira.mira.models.utils_time_normalization import normalize_time_for_ctrope

# 1. SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "./checkpoints"  # Path to your fine-tuned model
# model = MIRAForPrediction.from_pretrained(model_path).to(device)
model.eval()

# The 17 items identified for your project
VALID_ITEMS = [
    "220052", "220045", "220210", "220277", "223762", "221906", "221289", 
    "222315", "221662", "221749", "50813", "50912", "50820", "50885", 
    "51265", "51301", "50931"
]

def generate_mira_results(input_jsonl, output_jsonl, steps=3):
    """
    Runs MIRA inference and saves results for the Symbolic Judge.
    """
    results_count = 0
    done = []
    
    with open(input_jsonl, 'r') as f_in, open(output_jsonl, 'w') as f_out:
        for line in f_in:
            data = json.loads(line)
            itemid = str(data.get('itemid'))
            if itemid in done:
                continue
            if itemid not in VALID_ITEMS:
                continue
            done.append(itemid)
            # Prepare Tensors
            seq = torch.tensor([data['sequence']], dtype=torch.float32).to(device)
            times = torch.tensor([data['time']], dtype=torch.float32).to(device)
            
            # Simple Normalization (Standard for MIRA training)
            mean, std = seq.mean(), seq.std() + 1e-6
            seq_norm = (seq - mean) / std

            # Time Normalization for CT-RoPE
            full_scaled_times, _, _ = normalize_time_for_ctrope(
                time_values=times,
                attention_mask=torch.ones_like(times),
                seq_length=seq.shape[1],
                alpha=1.0
            )

            # Autoregressive Forecasting Loop
            current_seq = seq_norm.clone()
            current_times = full_scaled_times.clone()
            forecast_norm = []

            with torch.no_grad():
                for _ in range(steps):
                    output = model(input_ids=current_seq.unsqueeze(-1), time_values=current_times)
                    next_val = output.logits[:, -1, :]
                    forecast_norm.append(next_val.item())
                    
                    # Update for next step (t + 1 hour)
                    current_seq = torch.cat([current_seq, next_val], dim=1)
                    next_t = current_times[:, -1:] + 1.0 
                    current_times = torch.cat([current_times, next_t], dim=1)

            # De-normalize to original scale
            forecast_real = [(p * std.item()) + mean.item() for p in forecast_norm]

            # Create entry for Symbolic Judge
            result_entry = {
                "itemid": itemid,
                "forecast": [round(v, 4) for v in forecast_real],
                # In a real pipeline, pull these from your EHR medication table
                "current_meds": data.get("current_meds", []) 
            }
            print(result_entry)
            
            f_out.write(json.dumps(result_entry) + '\n')
            results_count += 1

    print(f"✅ Generated {results_count} forecasts in {output_jsonl}")

# 2. RUN
# Use a subset of your test data to save time for the deadline
generate_mira_results("train_mira.jsonl", "mira_test_results.jsonl")

C:\Users\nisha\AppData\Local\Temp\ipykernel_27872\4204207967.py:41: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  mean, std = seq.mean(), seq.std() + 1e-6


{'itemid': '50885', 'forecast': [nan, nan, nan], 'current_meds': []}
{'itemid': '50912', 'forecast': [0.9206, 0.9356, 0.9417], 'current_meds': []}
{'itemid': '50931', 'forecast': [121.6776, 123.6276, 124.4182], 'current_meds': []}
{'itemid': '51265', 'forecast': [252.8538, 254.2037, 254.7511], 'current_meds': []}
{'itemid': '51301', 'forecast': [22.38, 22.2543, 22.2354], 'current_meds': []}
{'itemid': '220045', 'forecast': [81.2747, 82.3321, 84.7205], 'current_meds': []}
{'itemid': '220210', 'forecast': [21.3483, 19.0261, 18.5221], 'current_meds': []}
{'itemid': '220277', 'forecast': [98.2237, 98.3879, 98.4429], 'current_meds': []}
{'itemid': '50813', 'forecast': [nan, nan, nan], 'current_meds': []}
{'itemid': '50820', 'forecast': [nan, nan, nan], 'current_meds': []}
{'itemid': '220052', 'forecast': [82.9374, 85.6239, 87.6739], 'current_meds': []}
{'itemid': '221749', 'forecast': [1.8246, 1.6547, 1.4647], 'current_meds': []}
{'itemid': '221906', 'forecast': [0.0662, 0.0585, 0.0541], 'c

In [24]:
class SymbolicLayer:
    def __init__(self):
        #qSOFA and Sepsis Logic Implementation 
        self.rules = {
                # Vitals (Standard Thresholds)
                "220052": {"name": "MAP", "min": 65},          # Sepsis-3 threshold [cite: 44]
                "220045": {"name": "Heart Rate", "max": 100},  # Tachycardia [cite: 39]
                "220210": {"name": "Resp Rate", "max": 22},    # qSOFA criteria [cite: 28]
                "220277": {"name": "SpO2", "min": 90},         # Hypoxia threshold
                "223762": {"name": "Temp (C)", "range": (36, 38)}, 
                
                # Labs (Organ Failure / SOFA markers) [cite: 69, 133]
                "50813": {"name": "Lactate", "max": 2.0},      # Septic Shock marker
                "50820": {"name": "pH", "min": 7.35},          # Acidosis
                "50912": {"name": "Creatinine", "max": 1.2},   # Renal SOFA
                "50885": {"name": "Bilirubin", "max": 1.2},    # Hepatic SOFA
                "51265": {"name": "Platelets", "min": 150},    # Coagulation SOFA
                "51301": {"name": "WBC", "range": (4, 12)},    # Infection marker
                "50931": {"name": "Glucose", "range": (70, 180)}
            }
            
            # Pressor IDs for Refractory Shock Logic [cite: 19, 44]
        self.pressors = ["221289", "221662", "221749", "221906", "222315"]
        
    def check_violation(self, itemid, forecast_val, patient_meds=None):
        """Returns 1 if a clinical law is violated, else 0."""
        rule = self.rules.get(itemid)
        if not rule: return 0
        
        violation = 0
        # Basic threshold checks
        if "min" in rule and forecast_val < rule["min"]: violation = 1
        if "max" in rule and forecast_val > rule["max"]: violation = 1
        if "range" in rule:
            if forecast_val < rule["range"][0] or forecast_val > rule["range"][1]:
                violation = 1
                
        # Advanced Logic: MAP < 65 while on Pressors indicates shock [cite: 19, 44]
        if itemid == "220052" and forecast_val < 65:
            if any(m in (patient_meds or []) for m in self.pressors):
                violation = 1 # High-risk refractory state detected
                
        return violation

In [25]:
def calculate_vcc(patient_results, judge):
    """
    Automates Vcc calculation across N patients.
    Vcc = Total Violations / Total Predictions.
    """
    total_predictions = 0
    total_violations = 0
    
    for record in patient_results:
        itemid = str(record['itemid'])
        forecasts = record['forecast'] # MIRA's statistical output 
        meds = record.get('current_meds', [])
        
        for val in forecasts:
            total_predictions += 1
            total_violations += judge.check_violation(itemid, val, meds)
            
    vcc_rate = total_violations / total_predictions if total_predictions > 0 else 0
    return vcc_rate, total_violations, total_predictions

In [26]:
def run_symbolic_pipeline(inference_file):
    judge = SymbolicLayer()
    
    # Assuming 'inference_file' is JSONL from your MIRA forecasting run
    with open(inference_file, 'r') as f:
        data = [json.loads(line) for line in f]
    
    vcc, v_count, p_count = calculate_vcc(data, judge)
    
    print(f"--- Neurosymbolic Evaluation Results ---")
    print(f"Total Forecasted Points: {p_count}")
    print(f"Total Clinical Violations: {v_count}")
    print(f"Constraint Violation Rate (Vcc): {vcc:.4f}")
    print(f"Interpretation: {vcc*100:.1f}% of statistical forecasts violate medical logic.")

In [28]:
run_symbolic_pipeline("mira_test_results.jsonl")

--- Neurosymbolic Evaluation Results ---
Total Forecasted Points: 6315
Total Clinical Violations: 968
Constraint Violation Rate (Vcc): 0.1533
Interpretation: 15.3% of statistical forecasts violate medical logic.


In [13]:
import pandas as pd
import networkx as nx

class HierarchicalGraphRAG:
    """
    Implements the two-tier agent architecture from the proposed framework.
    Uses PrimeKG to provide biological grounding for MIRA forecasts.
    """
    def __init__(self, primekg_csv_path):
        print("🔍 Initializing Hierarchical Agent System with PrimeKG...")
        # Load PrimeKG using the headers you provided
        df = pd.read_csv(primekg_csv_path, low_memory=False)
        
        # We use x_name and y_name as nodes, and display_relation as the edge attribute
        self.graph = nx.from_pandas_edgelist(
            df, 
            source='x_name', 
            target='y_name', 
            edge_attr=['display_relation']
        )
        
        # Mapping your clinical itemids to x_name/y_name values in PrimeKG [cite: 94, 130]
#         self.id_to_primekg = {
#     "220052": "MAP",
#     "220045": "Heart Rate",
#     "220210": "Resp Rate",
#     "220277": "SpO2",
#     "223762": "Temp (C)",
#     "50813": "Lactate",
#     "50820": "pH",
#     "50912": "Creatinine",
#     "50885": "Bilirubin",
#     "51265": "Platelets",
#     "51301": "WBC",
#     "50931": "Glucose"
# }
        self.id_to_primekg = {
            "220052": "PHYHIP",
            "220045": "GPANK1",
            "220210": "ZRSR2"
        }

    def tier_1_gp_agent(self, patient_metadata):
        """
        GP Agent: Identifies broad clinical context[cite: 79].
        """
        if "sepsis" in patient_metadata.lower():
            return "Sepsis"
        return "Systemic Inflammation"

    def tier_2_specialist_agent(self, signal_name, condition_context):
        """
        Specialist Agent: Retrieves niche subgraphs to aid MoE routing[cite: 20, 79].
        """
        try:
            # Find the semantic path representing clinical reasoning [cite: 38]
            path = nx.shortest_path(self.graph, source=signal_name, target=condition_context)
            
            subgraph = []
            for i in range(len(path) - 1):
                rel = self.graph[path[i]][path[i+1]]['display_relation']
                subgraph.append(f"({path[i]}) --[{rel}]--> ({path[i+1]})")
            return subgraph
        except (nx.NetworkXNoPath, KeyError):
            return ["No verifiable path found in PrimeKG."]

    def run_retrieval(self, itemid, patient_meta):
        signal_name = self.id_to_primekg.get(itemid)
        context = self.tier_1_gp_agent(patient_meta)
        subgraph = self.tier_2_specialist_agent(signal_name, context)

        return {
            "signal": signal_name,
            "subgraph": subgraph
        }



In [14]:
# --- TEST ---
rag_engine = HierarchicalGraphRAG("kg.csv")
print(rag_engine.run_retrieval("220052", "Patient has suspected sepsis"))

🔍 Initializing Hierarchical Agent System with PrimeKG...
{'signal': 'PHYHIP', 'subgraph': ['(PHYHIP) --[ppi]--> (DYRK1A)', '(DYRK1A) --[ppi]--> (FOXP3)', '(FOXP3) --[associated with]--> (immune dysregulation-polyendocrinopathy-enteropathy-X-linked syndrome)', '(immune dysregulation-polyendocrinopathy-enteropathy-X-linked syndrome) --[phenotype present]--> (Sepsis)']}


In [2]:
import torch

# 1. Setup Data (Assume Batch=1 for now)
# vitals: [1, 48, 5] (Batch, Seq, Features)
# times: [1, 48]
vitals_tensor = torch.randn(1, 48, 5).to("cuda")
times_tensor = torch.arange(48).float().unsqueeze(0).to("cuda")

# 2. THE FIX: Reshape to Channel-Independent format
# Permute to [1, 5, 48], then reshape to [5, 48, 1]
# This makes each of your 5 vitals a 'separate' sequence for MIRA
B, L, C = vitals_tensor.shape
vitals_reshaped = vitals_tensor.permute(0, 2, 1).reshape(B * C, L, 1)

# Expand times to match the new batch size [5, 48]
times_expanded = times_tensor.expand(B * C, L)

# 3. Extract Hidden States from MIRA
with torch.no_grad():
    outputs = model(
        input_ids=vitals_reshaped, 
        time_values=times_expanded,
        output_hidden_states=True
    )
    
    # last_hidden_state shape: [5, 48, 384] (assuming hidden_dim is 384)
    last_hidden_state = outputs.hidden_states[-1]
    
    # 4. POOLING: Get the "Clinical State"
    # First, take the last timestep of each variable: [5, 384]
    variable_latents = last_hidden_state[:, -1, :] 
    
    # Second, Mean-Pool across the 5 variables to get a single PATIENT vector
    # Shape: [1, 384] -> THIS IS YOUR LATENT VECTOR for the MoE Router
    latent_vector = variable_latents.mean(dim=0, keepdim=True)

print(f"Final Latent Vector Shape for MoE: {latent_vector.shape}")

Final Latent Vector Shape for MoE: torch.Size([1, 384])


In [5]:
import pandas as pd
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

def build_static_index(kg_path, out_prefix="primekg"):
    print("🏗️ Loading PrimeKG and generating embeddings...")
    df = pd.read_csv(kg_path, low_memory=False)
    
    # Extract unique nodes from both x_name and y_name
    nodes = pd.concat([df['x_name'], df['y_name']]).unique().astype(str)
    
    # 1. Generate Embeddings (Use a fast local model)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(nodes, show_progress_bar=True, batch_size=128)
    
    # 2. Save the FAISS Index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype('float32'))
    faiss.write_index(index, f"{out_prefix}.index")
    
    # 3. Save the Node List (to map indices back to names)
    with open(f"{out_prefix}_nodes.pkl", "wb") as f:
        pickle.dump(nodes.tolist(), f)
        
    print(f"✅ Static Index Saved: {out_prefix}.index and {out_prefix}_nodes.pkl")

if __name__ == "__main__":
    build_static_index("kg.csv")

🏗️ Loading PrimeKG and generating embeddings...


c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches:   0%|          | 0/1 [00:00<?, ?it/s]


ValueError: Unsupported input type: ArrowStringArray. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file
from main import NeurosymbolicPipeline
PIPELINE = NeurosymbolicPipeline(
    mira_ckpt="mira/checkpoints",
    kg_path="kg.csv",
    gemini_key=os.getenv("GEMINI_API")
)

    # Test with a sample from your MIMIC-IV dataset


🛠️ Initializing Full Neurosymbolic Pipeline...
⚡ Loading Pre-computed Knowledge Index...


c:\Users\nisha\OneDrive\Desktop\Clones\graphrag-timeseries\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Graph-RAG Ready.
